# Phase 3 · Exploration and cleaning of Olist

Tables covered in this part of the project: `order_items`, `products`, `sellers`, and `category_translation`. The `orders` table is accessed solely to verify references from the items. Run the notebook from the project root, with the CSV files from [Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) placed in the `datos/` directory.

In [7]:
from decimal import Decimal
from pathlib import Path

import pandas as pd
from IPython.display import display

data_dir = Path("datos")
input_files = {
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}
primary_keys = {
    "order_items": ["order_id", "order_item_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"],
}
raw = {
    name: pd.read_csv(data_dir / filename, dtype="string", keep_default_na=False, na_values=[""])
    for name, filename in input_files.items()
}

## 1. Rows, columns, gaps, and duplicates

Each row in `order_items` represents an item within an order. Its key is the pair `(order_id, order_item_id)`. The other tables are identified by product, seller, or the Portuguese name of the category. The CSV files are initially loaded as text to inspect their format before converting the numeric and date fields.

In [8]:
overview = []
column_profiles = {}
for name, table in raw.items():
    blank_cells = table.apply(lambda column: column.str.fullmatch(r"\s*").sum())
    overview.append({
        "tabla": name,
        "filas": len(table),
        "columnas": len(table.columns),
        "celdas nulas": int(table.isna().sum().sum()),
        "solo espacios": int(blank_cells.sum()),
        "filas duplicadas": int(table.duplicated().sum()),
        "claves repetidas": int(table.duplicated(primary_keys[name]).sum()),
    })
    column_profiles[name] = pd.DataFrame({
        "nulos": table.isna().sum(),
        "solo espacios": blank_cells,
        "valores distintos": table.nunique(),
    })

display(pd.DataFrame(overview).set_index("tabla"))
display(pd.concat(column_profiles, names=["tabla", "columna"]))

,filas,columnas,celdas nulas,solo espacios,filas duplicadas,claves repetidas
tabla,,,,,,
order_items,112650,7,0,0,0,0
products,32951,9,2448,0,0,0
sellers,3095,4,0,0,0,0
category_translation,71,2,0,0,0,0


nulos  solo espacios  \
tabla                columna                                               
order_items          order_id                           0              0   
                     order_item_id                      0              0   
                     product_id                         0              0   
                     seller_id                          0              0   
                     shipping_limit_date                0              0   
                     price                              0              0   
                     freight_value                      0              0   
products             product_id                         0              0   
                     product_category_name            610              0   
                     product_name_lenght              610              0   
                     product_description_lenght       610              0   
                     product_photos_qty               610              0   
                     product_weight_g                   2              0   
                     product_length_cm                  2              0   
                     product_height_cm                  2              0   
                     product_width_cm                   2              0   
sellers              seller_id                          0              0   
                     seller_zip_code_prefix             0              0   
                     seller_city                        0              0   
                     seller_state                       0              0   
category_translation product_category_name              0              0   
                     product_category_name_english      0              0   

                                                    valores distintos  
tabla                columna                                           
order_items          order_id                                   98666  
                     order_item_id                                 21  
                     product_id                                 32951  
                     seller_id                                   3095  
                     shipping_limit_date                        93318  
                     price                                       5968  
                     freight_value                               6999  
products             product_id                                 32951  
                     product_category_name                         73  
                     product_name_lenght                           66  
                     product_description_lenght                  2960  
                     product_photos_qty                            19  
                     product_weight_g                            2204  
                     product_length_cm                             99  
                     product_height_cm                            102  
                     product_width_cm                              95  
sellers              seller_id                                   3095  
                     seller_zip_code_prefix                      2246  
                     seller_city                                  611  
                     seller_state                                  23  
category_translation product_category_name                         71  
                     product_category_name_english                 71

## 2. Keys, prices, and product fields

Leading and trailing spaces are removed from text fields, and exact duplicates are eliminated. If distinct rows share the same key, execution is halted so they can be reviewed. The downloaded file contains no duplicate rows across the four tables.

Monetary values ​​are validated and converted into precise decimals. A freight value of zero is considered valid. For products, empty dimension and weight fields are preserved, and weights of zero are retained. The original headers are kept to facilitate subsequent data loading.

In [9]:
clean = {}
for name, table in raw.items():
    clean[name] = (
        table.apply(lambda column: column.str.strip().replace("", pd.NA))
        .drop_duplicates()
        .copy()
    )
    keys = clean[name][primary_keys[name]]
    if keys.isna().any().any() or keys.duplicated().any():
        raise ValueError(f"{name} contiene claves primarias nulas o repetidas.")

items = clean["order_items"]
items["order_item_id"] = items["order_item_id"].astype("Int64")
if items["order_item_id"].lt(1).any():
    raise ValueError("order_item_id debe ser positivo.")

items["shipping_limit_date"] = pd.to_datetime(
    items["shipping_limit_date"], format="%Y-%m-%d %H:%M:%S", errors="raise"
)
if items["shipping_limit_date"].isna().any():
    raise ValueError("Hay artículos sin fecha límite de despacho.")

money_summary = []
for column in ("price", "freight_value"):
    if not items[column].str.fullmatch(r"[0-9]+(?:\.[0-9]{1,2})?", na=False).all():
        raise ValueError(f"{column} contiene importes vacíos o con formato inválido.")
    items[column] = items[column].map(lambda value: Decimal(value).quantize(Decimal("0.01")))
    money_summary.append({
        "campo": column,
        "mínimo": items[column].min(),
        "máximo": items[column].max(),
        "ceros": int(items[column].eq(0).sum()),
    })
display(pd.DataFrame(money_summary).set_index("campo"))

products = clean["products"]
product_number_columns = [
    "product_name_lenght", "product_description_lenght", "product_photos_qty",
    "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
]
products[product_number_columns] = products[product_number_columns].astype("Int64")
display(pd.DataFrame({
    "nulos": products[product_number_columns].isna().sum(),
    "ceros": products[product_number_columns].eq(0).sum(),
}))

sellers = clean["sellers"]
if not sellers["seller_zip_code_prefix"].str.fullmatch(r"[0-9]{5}", na=False).all():
    raise ValueError("Se encontraron prefijos postales de vendedor distintos de cinco dígitos.")

,mínimo,máximo,ceros
campo,,,
price,0.85,6735.00,0
freight_value,0.00,409.68,383


,nulos,ceros
product_name_lenght,610,0
product_description_lenght,610,0
product_photos_qty,610,0
product_weight_g,2,4
product_length_cm,2,0
product_height_cm,2,0
product_width_cm,2,0


## 3. Product Categories in English

Category names left empty by Olist are retained as null values. For existing category names in Portuguese, the catalog is supplemented with two manual translations. These translations are stored in `category_translation` so they can be joined with the `products` dataset using `product_category_name`. The label `unknown` is used for visualization purposes when grouping these products in queries.

In [10]:
category_key = "product_category_name"
english_name = "product_category_name_english"
categories = clean["category_translation"]
if categories[english_name].isna().any():
    raise ValueError("El catálogo contiene traducciones vacías.")

has_category = products[category_key].notna()
needs_translation = has_category & ~products[category_key].isin(categories[category_key])
missing = products.loc[needs_translation, category_key].value_counts()
manual_translations = {
    "pc_gamer": "gaming_pc",
    "portateis_cozinha_e_preparadores_de_alimentos":
        "portable_kitchen_appliances_and_food_preparation",
}
additions = missing.rename("productos").to_frame()
additions[english_name] = additions.index.map(manual_translations)
if additions[english_name].isna().any():
    raise ValueError("Quedan categorías con nombre pendientes de traducir.")

clean["category_translation"] = pd.concat(
    [categories, additions.reset_index()[[category_key, english_name]]],
    ignore_index=True,
)
display(additions)

product_categories = products[["product_id", category_key]].merge(
    clean["category_translation"],
    on=category_key,
    how="left",
    validate="many_to_one",
)
if product_categories.loc[product_categories[category_key].notna(), english_name].isna().any():
    raise ValueError("Hay categorías conocidas sin traducción tras el cruce.")
print(f"Productos sin categoría de origen: {(~has_category).sum():,}")
print(f"Productos con traducción añadida: {needs_translation.sum():,}")

,productos,product_category_name_english
product_category_name,,
portateis_cozinha_e_preparadores_de_alimentos,10,portable_kitchen_appliances_and_food_preparation
pc_gamer,3,gaming_pc


Productos sin categoría de origen: 610
Productos con traducción añadida: 13


## 4. Integrity and Export

It is verified that the items point to existing orders, products, and sellers, and that every category specified for the products has a corresponding entry in the catalog. The results are saved—retaining the original names—in `data/clean/`. The Kaggle CSV files remain in `data/`.

In [11]:
order_ids = pd.read_csv(
    data_dir / "olist_orders_dataset.csv", usecols=["order_id"], dtype="string"
)["order_id"]
missing_references = {
    "order_items → orders": int((~items["order_id"].isin(order_ids)).sum()),
    "order_items → products": int((~items["product_id"].isin(products["product_id"])).sum()),
    "order_items → sellers": int((~items["seller_id"].isin(sellers["seller_id"])).sum()),
    "products → category_translation": int(
        (has_category & ~products[category_key].isin(clean["category_translation"][category_key])).sum()
    ),
}
display(pd.Series(missing_references, name="referencias sin coincidencia").to_frame())
if any(missing_references.values()):
    raise ValueError("Hay referencias sin coincidencia. Revisa las tablas antes de exportar.")

clean_dir = data_dir / "limpios"
clean_dir.mkdir(exist_ok=True)
exported = []
for name, filename in input_files.items():
    table = clean[name]
    target = clean_dir / filename
    table.to_csv(target, index=False, encoding="utf-8", date_format="%Y-%m-%d %H:%M:%S")
    exported.append({
        "tabla": name,
        "filas de origen": len(raw[name]),
        "filas exportadas": len(table),
        "archivo": target.as_posix(),
    })

display(pd.DataFrame(exported).set_index("tabla"))

,referencias sin coincidencia
order_items → orders,0
order_items → products,0
order_items → sellers,0
products → category_translation,0


,filas de origen,filas exportadas,archivo
tabla,,,
order_items,112650,112650,datos/limpios/olist_order_items_dataset.csv
products,32951,32951,datos/limpios/olist_products_dataset.csv
sellers,3095,3095,datos/limpios/olist_sellers_dataset.csv
category_translation,71,73,datos/limpios/product_category_name_translatio...
